# Baseline canonique — résultats officiels

Ce notebook est une lecture courte et sans recalcul des artefacts produits par le protocole commun. Il n'entraîne rien, ne réévalue rien et n'écrit aucun fichier.

## Sources officielles

- `scripts/evaluation_protocol.py` : split, cas d'évaluation et métriques.
- `scripts/Models/evaluate_final.py` : définition et évaluation de la baseline.
- `results/eval_baseline.json` : bundle officiel détaillé, lu uniquement.
- `results/final_evaluation.json` : synthèse finale, lue uniquement.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "results" / "final_evaluation.json").exists()
)

with (ROOT / "results" / "eval_baseline.json").open(encoding="utf-8") as stream:
    baseline_bundle = json.load(stream)
with (ROOT / "results" / "final_evaluation.json").open(encoding="utf-8") as stream:
    final_evaluation = json.load(stream)

print("Artefacts officiels chargés en lecture seule.")

Artefacts officiels chargés en lecture seule.


## Définition de la baseline

Pour chaque requête, les candidats partagent **au moins un genre** avec les seeds et ont reçu **au moins 5 notes dans train + validation**. Ils sont triés par **note moyenne décroissante**, puis par **identifiant croissant** en cas d'égalité. **Seuls les films seeds sont exclus** : ni historique utilisateur complet, ni popularité, ni note des seeds n'intervient.

In [2]:
protocol = baseline_bundle["protocol"]
protocol_view = pd.Series({
    "split": f"{protocol['train_frac']:.0%} train / {protocol['validation_frac']:.0%} validation / {protocol['test_frac']:.0%} test",
    "référence": protocol["reference_split"],
    "held-out": protocol["heldout_split"],
    "définition aimé": protocol["liked_definition"],
    "sélection seeds": protocol["seed_selection"],
    "seeds max": protocol["max_seeds"],
    "cas test": baseline_bundle["n_cases"],
}, name="Valeur").to_frame()
display(protocol_view)

,Valeur
split,65% train / 15% validation / 20% test
référence,train+validation
held-out,test
définition aimé,rating > user_mean(reference)
sélection seeds,deterministic_random_1_to_3
seeds max,3
cas test,921


## Protocole, split et cas test

Le split déterministe commun est **65 % train / 15 % validation / 20 % test** (`random_state=42`). La référence est **train + validation** et le test reste held-out. Un film est aimé si sa note est strictement supérieure à la moyenne utilisateur calculée sur la référence. Chaque cas utilise **1 à 3 seeds tirées aléatoirement de façon déterministe**, sans biais de note ou de popularité. P@5 et P@10 utilisent un dénominateur fixe.

In [3]:
method = baseline_bundle["methods"]["baseline_overlap_mean"]
sample_cases = pd.DataFrame({
    "cas officiel": range(1, 4),
    "recommandations top 5": method["recs_top5"][:3],
    "P@5": method["P@5"][:3],
    "P@10": method["P@10"][:3],
})
display(sample_cases)

,cas officiel,recommandations top 5,P@5,P@10
0,1,"[1449, 1524, 483, 64, 318]",0.0,0.1
1,2,"[169, 408, 12, 603, 480]",0.0,0.1
2,3,"[1449, 1524, 169, 483, 64]",0.0,0.1


## Résultat final

Les scores ci-dessous proviennent directement de `results/final_evaluation.json`. Le Hit Rate@5 est la part des cas ayant au moins un film aimé dans le top 5. Aucune moyenne n'est recalculée dans ce notebook.

In [4]:
scores = final_evaluation["scores"]["baseline_overlap_mean"]
result = pd.DataFrame(
    [{
        "méthode": "baseline_overlap_mean",
        "P@5": scores["P@5"],
        "P@10": scores["P@10"],
        "Hit Rate@5": scores["HitRate@5"],
    }]
).set_index("méthode")
display(
    result.style.format(
        {"P@5": "{:.4f}", "P@10": "{:.4f}", "Hit Rate@5": "{:.1%}"}
    )
)

,P@5,P@10,Hit Rate@5
méthode,,,
baseline_overlap_mean,0.0343,0.0372,15.5%


## Conclusion

In [5]:
print(
    f"Sur {final_evaluation['n_cases']} cas test communs, la baseline atteint "
    f"P@5={scores['P@5']:.4f}, P@10={scores['P@10']:.4f} et "
    f"Hit Rate@5={scores['HitRate@5']:.1%}. "
    "Elle constitue le point de comparaison canonique des modèles NextWatch."
)

Sur 921 cas test communs, la baseline atteint P@5=0.0343, P@10=0.0372 et Hit Rate@5=15.5%. Elle constitue le point de comparaison canonique des modèles NextWatch.


<!-- Cellule historique neutralisée. -->